In [ ]:
"""
REFERENCE: Pipeline + ColumnTransformer + Why Leakage Happens
"""

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier


# PART 1: THE CORE RULE TO REMEMBER

# ANY transformer that has a .fit() method (SimpleImputer, StandardScaler,
# OneHotEncoder, feature selectors, etc.) calculates some number(s) from
# whatever data you hand it in that exact .fit() call.
#
# It has NO memory of anything before, and NO awareness of "train" vs "test"
# as concepts -- it just sees "here is some data, calculate stats from it."
#
# THE LEAK HAPPENS WHEN: you calculate those stats (median, mean/std,
# categories) using data that includes rows that will later be used as
# "unseen" validation/test rows.

# PART 2: WHY THIS MATTERS EVEN AFTER YOU'VE ALREADY SPLIT CORRECTLY

# You already know to do: X_train, X_test = train_test_split(X, y)
# That's correct. But cross_val_score() takes X_train and SPLITS IT AGAIN,
# internally, into several folds (e.g. cv=5 means 5 folds).
#
# The danger: if you calculate median/mean/std/categories ONCE on the full
# X_train BEFORE calling cross_val_score, then every fold's "validation"
# rows (which are a subset of X_train) already influenced that calculation.
#
# Below is a small proof you can re-run anytime to convince yourself.


print("=" * 70)
print("PART 2 DEMO: proving the leak is real, with real numbers")
print("=" * 70)

# Tiny fake dataset: 10 rows, one column, one missing value
df = pd.DataFrame({"age": [20, 22, 25, 28, 30, np.nan, 35, 40, 60, 65]})

# Step 0: split FIRST (this part you already do correctly)
X_train, X_test = train_test_split(df, test_size=0.2, random_state=42)

# ---- THE WRONG WAY ----
# Calculating median ONCE, on the whole X_train, before cross-validation
wrong_median = X_train["age"].median()
print(f"\n[WRONG] Median computed once on full X_train: {wrong_median}")
print("This SAME number gets reused for every single fold below,")
print("even though each fold's validation rows should be 'unseen'.\n")

kf = KFold(n_splits=3, shuffle=False)
for fold_num, (train_idx, val_idx) in enumerate(kf.split(X_train), start=1):
    fold_val = X_train.iloc[val_idx]
    print(f"  Fold {fold_num} validation rows: {list(fold_val['age'].values)}"
          f"  <- these rows helped compute {wrong_median}, yet we're about"
          f" to 'test' on them using stats they influenced")

# ---- THE RIGHT WAY ----
# Recompute the median SEPARATELY inside each fold, using ONLY that fold's
# training rows (this is exactly what a Pipeline does automatically).
print("\n[RIGHT] Recomputing median fresh, per fold, using only that fold's train rows:")
for fold_num, (train_idx, val_idx) in enumerate(kf.split(X_train), start=1):
    fold_train = X_train.iloc[train_idx]
    fold_val = X_train.iloc[val_idx]
    correct_median = fold_train["age"].median()  # only from THIS fold's train rows
    print(f"  Fold {fold_num}: train rows = {list(fold_train['age'].values)}")
    print(f"           -> median computed = {correct_median}  "
          f"(validation rows {list(fold_val['age'].values)} had ZERO influence)")



# PART 3: WHAT PIPELINE ACTUALLY IS

# A Pipeline is just a conveyor belt: data goes in one end, passes through
# steps IN ORDER, comes out ready for the model.
#
# Each step is a tuple: (name_you_choose, transformer_object)
# The name is just a label -- it has no special meaning, you pick it.
#
# Once built, the WHOLE Pipeline behaves like a single object with its own
# .fit() / .transform() / .predict() -- even though internally it's doing
# several operations.


numeric_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="median")),        # fill missing numeric values
    ("scale", StandardScaler()),                          # mean=0, std=1
])

categorical_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),  # fill missing categories with mode
    ("ohe", OneHotEncoder(handle_unknown="ignore")),       # turn categories into 0/1 columns
    # handle_unknown="ignore" -> if test set has a category never seen in
    # train, it won't crash; it just encodes that row as all-zeros.
])



# PART 4: WHAT COLUMNTRANSFORMER ACTUALLY IS

# Numeric columns and categorical columns need DIFFERENT treatment.
# You cannot median-impute a text column, and you cannot one-hot-encode a
# number in the same way.
#
# ColumnTransformer's job: send each group of columns to its OWN pipeline,
# then glue all the outputs back together side-by-side into one final table.
#
# Structure: (name_you_choose, pipeline_to_use, list_of_columns_it_applies_to)


num_cols = ["age"]
cat_cols = ["city"]

preprocess = ColumnTransformer([
    ("num", numeric_pipeline, num_cols),        # age column -> numeric_pipeline
    ("cat", categorical_pipeline, cat_cols),    # city column -> categorical_pipeline
])



# PART 5: GLUE PREPROCESSING + MODEL INTO ONE PIPELINE OBJECT

# This is the step that makes leakage structurally impossible.
# Now "model" is ONE object. Calling model.fit(data) triggers the ENTIRE
# chain: impute -> scale -> encode -> train classifier, all using only
# whatever "data" was passed into THIS specific .fit() call.


model = Pipeline([
    ("prep", preprocess),                        # all preprocessing (numeric + categorical)
    ("clf", RandomForestClassifier(random_state=42)),  # the actual model
])



# PART 6: WHY cross_val_score(model, ...) IS SAFE

# cross_val_score does exactly what our manual KFold loop did in Part 2,
# but hidden inside one function call:
#
#   for each fold:
#       1. split X_train into fold_train / fold_val
#       2. call model.fit(fold_train, y_fold_train)
#          -> this re-triggers imputer.fit(), scaler.fit(), encoder.fit()
#             using ONLY fold_train -- never fold_val
#       3. call model.score(fold_val, y_fold_val) using those fold-specific
#          fitted values
#       4. repeat for the next fold
#
# None of steps 1-4 are visible in your code. That's why it can feel like
# "nothing is happening" -- it's all hidden inside this one line:


print("\n" + "=" * 70)
print("PART 6 DEMO: full pipeline + columntransformer running end-to-end")
print("=" * 70)

# Small fake dataset with numeric + categorical + target
np.random.seed(0)
demo_df = pd.DataFrame({
    "age":  [22, 25, np.nan, 30, 35, 40, 28, np.nan, 50, 45, 33, 29],
    "city": ["Delhi", "Mumbai", "Delhi", "Bangalore", "Mumbai", "Delhi",
             "Bangalore", "Delhi", "Mumbai", "Bangalore", "Delhi", "Mumbai"],
    "salary_tier": [0, 1, 0, 1, 1, 2, 0, 1, 2, 1, 0, 1],
})

X = demo_df[["age", "city"]]
y = demo_df["salary_tier"]

# Step 0: split first, always
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

# Step 1-6: cross-validate the WHOLE pipeline (safe -- refits per fold)
scores = cross_val_score(model, X_train, y_train, cv=3, scoring="accuracy")
print("Per-fold CV accuracy:", scores)
print("Mean CV accuracy:", scores.mean())

# Step 7: only now, fit on the FULL training set
model.fit(X_train, y_train)

# Step 8: evaluate on the test set (touched for the very first time, here)
print("Final test accuracy:", model.score(X_test, y_test))


# PART 7: ONE-LINE SUMMARY

# "I split first, then wrap every preprocessing step -- imputation, scaling,
#  encoding -- and the model itself inside one Pipeline object, using a
#  ColumnTransformer to route numeric and categorical columns to their own
#  branches. This matters because cross-validation re-splits the training
#  data into folds internally, and calling .fit() on the whole Pipeline once
#  per fold guarantees every fold's statistics -- median, mean/std, category
#  list -- are recalculated using ONLY that fold's training rows, so no
#  fold's validation rows ever influence the numbers used to transform them."
